# 02  Download the article text

DATA 5500  Sara Choque Molina

The BigQuery export only provides the **link** to each article, not the content inside it.
This notebook opens every link and saves the article text that comes back (if there are some).

This is the slow part of the project. About 5,000 links with a 1-second pause between each one is roughly 2 hours.

## Block 1  Setup

In [1]:
import os
import time

import pandas as pd
import trafilatura

FOLDER = r"C:\Users\sara_\Desktop\College\DATA 5500-01 Senior Des\data bases"

IN_FILE = os.path.join(FOLDER, "02_articles_2021_tiktok.csv")
OUT_FILE = os.path.join(FOLDER, "articles_2021_text.csv")

PAUSE = 1.0        # seconds to wait between links, so I don't hammer the news sites
SAVE_EVERY = 100   # write the results to disk every 100 links, in case something crashes

print("Input file exists:", os.path.exists(IN_FILE))

Input file exists: True


## Block 2 Open the CSV and look at it

In [3]:
df = pd.read_csv(IN_FILE)

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

Rows: 4995
Columns: ['day', 'source', 'url', 'locations', 'category']


,day,source,url,locations,category
0,20211115,103gbfrocks.com,https://103gbfrocks.com/satanic-high-school-cl...,1#American#US#US##39.828175#-98.5795#US#1454;2...,student_threat
1,20211115,inforum.com,https://www.inforum.com/news/education/7282773...,NaN,student_threat
2,20211115,grandforksherald.com,https://www.grandforksherald.com/news/educatio...,NaN,student_threat
3,20211115,wvlkam.com,https://www.wvlkam.com/news/school-delays-dave...,NaN,student_threat
4,20211115,kmjnow.com,https://www.kmjnow.com/news/school-delays-dave...,NaN,student_threat


## Block 3 Make the list of links to download

Two things happen here:

1. The same link can show up more than once in GDELT, so I keep only one copy of each.
2. If I already ran this notebook before and it stopped halfway, I skip the links that
   are already saved in the output file. That way I never download the same thing twice.

In [4]:
all_urls = df["url"].dropna().unique().tolist()
print("Unique links in the CSV:", len(all_urls))

# Pick up where I left off, if there is already an output file
already_done = set()
if os.path.exists(OUT_FILE):
    already_done = set(pd.read_csv(OUT_FILE)["url"])
    print("Already downloaded in a previous run:", len(already_done))

todo = [u for u in all_urls if u not in already_done]
print("Links left to download:", len(todo))
print("Rough time:", round(len(todo) * PAUSE / 60), "minutes")

Unique links in the CSV: 4995
Links left to download: 4995
Rough time: 83 minutes


## Block 4 Download

This is the long one. It prints a line every 100 links so I can tell it is still alive.

What the two trafilatura commands do:

- `fetch_url` goes to the website and brings back the raw HTML, which is the page with all
  the menus, ads and junk still in it.
- `extract` throws away the junk and leaves only the article itself.

The whole thing is inside a `try` / `except` because plenty of these links are dead by now
(the articles are from 2021). Without it, one broken link would stop the entire run.

In [5]:
results = []
start = time.time()


def save():
    """Append whatever is in results to the output file, then empty results."""
    if not results:
        return
    out = pd.DataFrame(results)
    header_needed = not os.path.exists(OUT_FILE)
    out.to_csv(OUT_FILE, mode="a", header=header_needed, index=False)
    results.clear()


for i, url in enumerate(todo, start=1):
    text = ""
    note = "ok"

    try:
        html = trafilatura.fetch_url(url)
        if html is None:
            note = "link is dead"
        else:
            text = trafilatura.extract(html) or ""
            if text == "":
                note = "page opened but no article text"
    except Exception as e:
        note = "error: " + type(e).__name__

    results.append({
        "url": url,
        "words": len(text.split()),
        "note": note,
        "text": text,
    })

    if i % SAVE_EVERY == 0:
        save()
        minutes = (time.time() - start) / 60
        print(f"{i} of {len(todo)} done  ·  {minutes:.0f} minutes so far")

    time.sleep(PAUSE)

save()
print("Finished. Saved to:", OUT_FILE)

100 of 4995 done  ·  3 minutes so far
200 of 4995 done  ·  6 minutes so far
300 of 4995 done  ·  9 minutes so far
400 of 4995 done  ·  12 minutes so far
500 of 4995 done  ·  15 minutes so far
600 of 4995 done  ·  18 minutes so far
700 of 4995 done  ·  21 minutes so far
800 of 4995 done  ·  25 minutes so far
900 of 4995 done  ·  28 minutes so far
1000 of 4995 done  ·  32 minutes so far
1100 of 4995 done  ·  35 minutes so far
1200 of 4995 done  ·  38 minutes so far
1300 of 4995 done  ·  41 minutes so far
1400 of 4995 done  ·  44 minutes so far
1500 of 4995 done  ·  47 minutes so far
1600 of 4995 done  ·  50 minutes so far
1700 of 4995 done  ·  54 minutes so far
1800 of 4995 done  ·  57 minutes so far
1900 of 4995 done  ·  60 minutes so far
2000 of 4995 done  ·  63 minutes so far
2100 of 4995 done  ·  66 minutes so far
2200 of 4995 done  ·  70 minutes so far
2300 of 4995 done  ·  73 minutes so far
2400 of 4995 done  ·  76 minutes so far
2500 of 4995 done  ·  79 minutes so far
2600 of 4995

## Block 5 — Check what I got

Numbers I need for the progress report: how many links worked, and how much text I ended up
with. Articles under 100 words are usually just a headline and a photo caption, so they are
not useful for training and I count them separately.

In [6]:
got = pd.read_csv(OUT_FILE)

worked = got[got["words"] > 0]
usable = got[got["words"] >= 100]

print("Links tried:        ", len(got))
print("Text came back:     ", len(worked), f"({len(worked) / len(got):.0%})")
print("Usable (100+ words):", len(usable), f"({len(usable) / len(got):.0%})")
print()
print("Median length of the ones that worked:", int(worked["words"].median()), "words")
print()
print("Why the rest failed:")
print(got[got["words"] == 0]["note"].value_counts().to_string())

Links tried:         4995
Text came back:      2822 (56%)
Usable (100+ words): 2649 (53%)

Median length of the ones that worked: 324 words

Why the rest failed:
note
link is dead                       1884
page opened but no article text     289


In [7]:
got[["url", "words", "note"]].to_csv(
    os.path.join(FOLDER, "articles_2021_linkstatus.csv"), index=False
)

## What comes next

Notebook 03 takes `articles_2021_text.csv`, removes the stories that are the same article
republished on a different website, and then the labeling starts.